In [24]:
import ast
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

%matplotlib inline

##### 1. Load the CSV with pandas, drop fully blank rows, and inspect shape, dtypes, and `.info()`.

In [25]:
df = pd.read_csv('E_Mcdonaldsdata.csv')

C:\Users\Shiva Thakur\AppData\Local\Temp\ipykernel_18708\3696745864.py:1: DtypeWarning: Columns (0: table, 1: heading, 2: subheading, 3: 2022, 4: 2021) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('E_Mcdonaldsdata.csv')


In [26]:
df.head()

,table,heading,subheading,2024,2023,2022,2021,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,revenue_breakdown,company_operated,company_operated_us,3197.0,3221.0,"2,836","2,617",NaN,NaN,NaN,NaN,NaN
1,revenue_breakdown,company_operated,company_operated_international_operated_markets,5713.0,5702.0,"5,179","6,456",NaN,NaN,NaN,NaN,NaN
2,revenue_breakdown,company_operated,company_operated_intl_dev_licensed_and_corp,872.0,819.0,733,715,NaN,NaN,NaN,NaN,NaN
3,revenue_breakdown,franchised,franchised_us,7211.0,7163.0,"6,585","6,094",NaN,NaN,NaN,NaN,NaN
4,revenue_breakdown,franchised,franchised_international_operated_markets,6746.0,6549.0,"5,985","5,638",NaN,NaN,NaN,NaN,NaN


In [27]:
print(df.columns.tolist())

['table', 'heading', 'subheading', '2024', '2023', '2022', '2021', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']


In [28]:
col = ['Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']
df.drop(columns=col, inplace= True)
print(df.columns.tolist())

['table', 'heading', 'subheading', '2024', '2023', '2022', '2021']


In [29]:
print('Shape of dataset is :', df.shape)
print('Type of dataset in table are :\n',df.dtypes)
df.info()

Shape of dataset is : (1048504, 7)
Type of dataset in table are :
 table             str
heading           str
subheading        str
2024          float64
2023          float64
2022              str
2021              str
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 1048504 entries, 0 to 1048503
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   table       85 non-null     str    
 1   heading     85 non-null     str    
 2   subheading  85 non-null     str    
 3   2024        85 non-null     float64
 4   2023        85 non-null     float64
 5   2022        85 non-null     str    
 6   2021        85 non-null     str    
dtypes: float64(2), str(5)
memory usage: 56.0 MB


In [30]:
df.describe()

,2024,2023
count,85.000000,85.000000
mean,2222.397882,2380.905882
std,13363.431265,12890.163342
min,-77375.000000,-74640.000000
25%,3.000000,12.000000
50%,636.000000,732.000000
75%,2536.000000,2886.000000
max,66834.000000,63480.000000


In [31]:
df.head()

,table,heading,subheading,2024,2023,2022,2021
0,revenue_breakdown,company_operated,company_operated_us,3197.0,3221.0,"2,836","2,617"
1,revenue_breakdown,company_operated,company_operated_international_operated_markets,5713.0,5702.0,"5,179","6,456"
2,revenue_breakdown,company_operated,company_operated_intl_dev_licensed_and_corp,872.0,819.0,733,715
3,revenue_breakdown,franchised,franchised_us,7211.0,7163.0,"6,585","6,094"
4,revenue_breakdown,franchised,franchised_international_operated_markets,6746.0,6549.0,"5,985","5,638"


##### 2. Clean the year columns — strip `$`, `,` and whitespace, then convert to `float`.

In [32]:
def cleanon_ast(val): #Defines a function that will be applied to one cell at a time (one value from your column).
    if pd.isna(val): #Checks if the value is missing (NaN). If it is, just return it as-is — don't try to clean or convert it. 
                     #This prevents an error later, since you can't strip $ and , from something that isn't a real string.
        return val

    clean_data = str(val).replace('$','').replace(',','').strip() 
    #This line does four things in sequence, left to right:
    #str(val) — converts the value to a string (in case it's already a number or some other type).
    #.replace('$', '') — removes every $ character.
    #.replace(',', '') — removes every comma (thousands separator).
    #.strip() — removes leading/trailing whitespace.
    #So " $1,234.50 " becomes "1234.50".
    
    return ast.literal_eval(clean_data) #convert that cleaned string into an actual int/float.

year_cols = ['2021','2022','2023','2024'] #Create a list of the column name you want to clean these should match your DataFrame real column names.

for col in year_cols:#Loop through that list one column name at a time, calling each one col.
    df[col] = df[col].apply(cleanon_ast) #For the current column (col), run clean_with_ast on every cell in that column 
                                         #and overwrite the column with the cleaned results.
                                         #This repeats for each column in year_cols

In [33]:
df.head()

,table,heading,subheading,2024,2023,2022,2021
0,revenue_breakdown,company_operated,company_operated_us,3197.0,3221.0,2836.0,2617.0
1,revenue_breakdown,company_operated,company_operated_international_operated_markets,5713.0,5702.0,5179.0,6456.0
2,revenue_breakdown,company_operated,company_operated_intl_dev_licensed_and_corp,872.0,819.0,733.0,715.0
3,revenue_breakdown,franchised,franchised_us,7211.0,7163.0,6585.0,6094.0
4,revenue_breakdown,franchised,franchised_international_operated_markets,6746.0,6549.0,5985.0,5638.0


##### 3. Pivot the long-format data (`table`, `heading`, `subheading`) into a wide table with one row per `subheading` and one column per year.

In [34]:
df = df.dropna(how='all')  # drop fully-empty rows
print(df.shape)

(85, 7)


In [35]:
wide_df = df.set_index('subheading')[['2021', '2022', '2023', '2024']]
print(wide_df)

                                                   2021    2022    2023  \
subheading                                                                
company_operated_us                              2617.0  2836.0  3221.0   
company_operated_international_operated_markets  6456.0  5179.0  5702.0   
company_operated_intl_dev_licensed_and_corp       715.0   733.0   819.0   
franchised_us                                    6094.0  6585.0  7163.0   
franchised_international_operated_markets        5638.0  5985.0  6549.0   
...                                                 ...     ...     ...   
cash_and_equivalents_increase_(decrease)         1260.0 -2126.0  1996.0   
cash_and_equivalents_at_beginning_of_year        3449.9  4709.0  2584.0   
cash_and_equivalents_at_end_of_year              4709.2  2584.0  4579.0   
interest_paid                                    1197.0  1184.0  1287.0   
income_taxes_paid                                2404.0  3024.0  2993.0   

                        

##### 4. Filter and display only rows where `table == 'revenue_breakdown'`.

In [36]:
df.head()

,table,heading,subheading,2024,2023,2022,2021
0,revenue_breakdown,company_operated,company_operated_us,3197.0,3221.0,2836.0,2617.0
1,revenue_breakdown,company_operated,company_operated_international_operated_markets,5713.0,5702.0,5179.0,6456.0
2,revenue_breakdown,company_operated,company_operated_intl_dev_licensed_and_corp,872.0,819.0,733.0,715.0
3,revenue_breakdown,franchised,franchised_us,7211.0,7163.0,6585.0,6094.0
4,revenue_breakdown,franchised,franchised_international_operated_markets,6746.0,6549.0,5985.0,5638.0


In [37]:
df[df['table'] == 'revenue_breakdown']

,table,heading,subheading,2024,2023,2022,2021
0,revenue_breakdown,company_operated,company_operated_us,3197.0,3221.0,2836.0,2617.0
1,revenue_breakdown,company_operated,company_operated_international_operated_markets,5713.0,5702.0,5179.0,6456.0
2,revenue_breakdown,company_operated,company_operated_intl_dev_licensed_and_corp,872.0,819.0,733.0,715.0
3,revenue_breakdown,franchised,franchised_us,7211.0,7163.0,6585.0,6094.0
4,revenue_breakdown,franchised,franchised_international_operated_markets,6746.0,6549.0,5985.0,5638.0
5,revenue_breakdown,franchised,franchised_intl_dev_licensed_and_corp,1758.0,1724.0,1536.0,1353.0
6,revenue_breakdown,other,other_revenues,423.0,316.0,329.0,351.0


##### 5. Find which `subheading` had the highest value in 2024.

In [38]:
df.head()

,table,heading,subheading,2024,2023,2022,2021
0,revenue_breakdown,company_operated,company_operated_us,3197.0,3221.0,2836.0,2617.0
1,revenue_breakdown,company_operated,company_operated_international_operated_markets,5713.0,5702.0,5179.0,6456.0
2,revenue_breakdown,company_operated,company_operated_intl_dev_licensed_and_corp,872.0,819.0,733.0,715.0
3,revenue_breakdown,franchised,franchised_us,7211.0,7163.0,6585.0,6094.0
4,revenue_breakdown,franchised,franchised_international_operated_markets,6746.0,6549.0,5985.0,5638.0


In [39]:
#clean one-liner showing both the subheading and its value

top_row2024 = df.loc[df['2024'].idxmax()]

#df.loc[..., 'subheading'] uses that index label to look up the subheading value at that row.
#df['2024'].idxmax() finds the index label of the row with the highest value in the 2024 column.

print(top_row2024[['subheading', '2024']])

subheading    retained_earnings
2024                    66834.0
Name: 57, dtype: object


In [41]:
top_row2023 = df.loc[df['2023'].idxmax()]

print(top_row2023[['heading', '2023']])

heading    shareholders
2023            63480.0
Name: 57, dtype: object


##### 6. Group by `heading` and sum values for each year (e.g., total company_operated vs franchised revenue).